[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/02_agent_loop_and_state.ipynb)


# Agentic Systems Foundations
## Notebook 02: The Agent Loop and State
**Duration:** 30 min &nbsp;|&nbsp; **Mode:** Demonstration + Guided Coding

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** **THINK → ACT → OBSERVE**, and the state update that closes the cycle.


In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Makes `agent_core` importable whether you are in Colab, in a local venv,
# or running from a clone. Installs nothing you do not need: the package's
# only hard dependency is the Python standard library.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


if IN_COLAB:
    # openai for the real provider; jsonschema + langchain for the parallel
    # mappings shown in notebook 03. All optional — the notebook degrades
    # gracefully if any is missing.
    _pip("openai", "python-dotenv", "jsonschema", "langchain-core")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

# Where the Acme data lives — the tools resolve this automatically, but we
# print it so a path problem is visible immediately rather than as an empty
# search result three cells later.
from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDER  (works with NO key at all)
# ============================================================
# Default stack = OpenAI gpt-4o-mini with native tool calling.
# In Colab the key is read from the SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON -> re-run this cell.
#
# With NO key we fall back to MockToolCallLLM. Read that name literally:
# unlike a text-only mock, it DECIDES TOOL CALLS, so the entire agent loop —
# every notebook in this session — runs offline and deterministically.
import os

def _load_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

HAS_KEY = _load_key()
os.environ.setdefault("AGENT_LLM_PROVIDER", "openai" if HAS_KEY else "mock")

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
if not HAS_KEY:
    print("\nNo API key found -> running on the offline mock.")
    print("Everything in this notebook still works. Outputs are labelled [mock].")

## WHY — state is the core of agent intelligence

Here is a claim worth being suspicious of, because it sounds like a slogan:

> **State management is the core of agent intelligence.**

It is not a slogan. It is the load-bearing fact, and here is the argument.

The model is *stateless*. Every API call is independent; it remembers nothing
between calls. So when your agent appears to "learn" that order ACME-1042 was
placed in July, nothing inside the model changed. What happened is that **you
appended that fact to the input** of the next call.

Which means: an agent's intelligence over time is exactly the quality of what
you write into its state, and nothing else. Get that wrong and no model, however
capable, can recover — because it will never see what it already did.


## WHAT — the loop, in full

```
    ┌─────────────────────────────────────────────┐
    │                                             │
    ▼                                             │
  STATE ──▶ THINK ──▶ ACT ──▶ OBSERVE ──▶ UPDATE ─┘
 (goal +    (LLM     (run a   (capture    (append to
  history)  decides)  tool)    result)     history)
    │
    └──▶ [terminate?] ──▶ ANSWER
```

And the code, which is genuinely this short:

```python
while not done:
    decision = llm.decide(state.messages, tools)     # THINK
    if decision.is_final:
        break                                        # terminate
    for call in decision.tool_calls:
        obs = tools.dispatch(call.name, call.args)   # ACT
        state.add_message("tool", str(obs.result))   # OBSERVE + UPDATE
```

**That is the entire idea of this session.** Six lines. Everything else —
schemas, skills, control, tracing — exists to keep those six lines honest.

### What state actually holds

| Field | Who reads it | Why it is separate |
|---|---|---|
| `goal` | you | immutable; the agent may not rewrite the task |
| `messages` | **the model** | the transcript sent on every call |
| `observations` | **your code** | structured; lets you ask "has it made progress?" without parsing prose |
| `budget` | the control layer | how much more it is allowed to do |

Beginners collapse `messages` and `observations` into one list, and then cannot
answer "is this agent stuck?" without re-reading English. Keep the
machine-readable record separate from the model-readable transcript.


## HOW (from scratch) — the whole agent, inline, in about 30 lines

In [ ]:
# Before using the package, let's write the loop by hand so there is no magic.
# This is a complete, working agent. Read every line.
from agent_core import get_llm
from agent_core.acme_tools import acme_registry

llm = get_llm()
tools = acme_registry().subset("get_order_status", "check_refund_eligibility")

goal = "Is order ACME-1046 refundable? The reason is changed_mind."

# ---- STATE: just a list of messages, plus a step counter -------------------
messages = [
    {"role": "system", "content": "You answer using tools. Gather evidence before answering."},
    {"role": "user", "content": goal},
]
step = 0
MAX_STEPS = 5

# ---- THE LOOP --------------------------------------------------------------
while step < MAX_STEPS:
    step += 1
    decision = llm.decide(messages, tools)                     # THINK

    if decision.is_final:                                      # terminate?
        print(f"\nstep {step}: FINAL")
        print(decision.content[:300])
        break

    for call in decision.tool_calls:                           # ACT
        obs = tools.dispatch(call.name, call.args, step=step)
        print(f"step {step}: {call.name}({call.args}) -> {obs.summary(70)}")

        # The provider protocol needs the request in the transcript before the
        # result, so append both.
        messages.append({
            "role": "assistant", "content": None,
            "tool_calls": [{"id": call.id, "type": "function",
                            "function": {"name": call.name,
                                         "arguments": str(call.args)}}],
        })
        messages.append({                                      # OBSERVE + UPDATE
            "role": "tool", "tool_call_id": call.id, "name": call.name,
            "content": str(obs.result) if obs.ok else f"ERROR: {obs.error}",
        })

print(f"\nTranscript grew from 2 messages to {len(messages)}.")
print("That growth IS the agent's memory.")

That is an agent. No framework, no base class, no magic — a `while` loop, a list
of dicts, and one function that decides what to do next.

The package version in `agent_core/loop.py` is the same loop with the sharp
edges filed off: it never lets a tool raise, it checks termination *before*
spending an LLM call, and it records a trace. Open it — it is under 60 lines of
actual code and you have now written most of it yourself.


> ### ✋ Predict before you run
> We are going to run the identical goal twice: once normally, and once with the OBSERVE step disabled — the tool still runs, but its result is never written back into the transcript. **What will the second run do?** Will it crash, answer wrongly, or something else? How many steps will it take?
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# THE EXPERIMENT THAT MAKES THE POINT.
# Identical agent, identical goal. The only difference: `stateful`.
from agent_core import Agent, compare

agent = Agent()
goal = "What is the status of order ACME-1048?"

with_state    = agent.run(goal, stateful=True)
without_state = agent.run(goal, stateful=False)

print(compare({"with state": with_state.trace, "WITHOUT state": without_state.trace}))
print()
print("with state    ->", with_state.state.stop_reason)
print("WITHOUT state ->", without_state.state.stop_reason)

**What you should observe:** the stateless agent calls the *same tool with the
same arguments*, over and over, until something stops it.

It is not broken and it is not stupid. It is behaving perfectly rationally: each
step it sees exactly the same input — the goal, and nothing else — so it makes
exactly the same decision. It has no way to know it already has the answer,
because nobody told it.

> **This is the single most important cell in the session.** Delete the state
> update and the loop still runs, still calls tools, still terminates. It just
> never learns anything. The `while` loop is not what makes an agent; **the
> write-back is.**

Note also *how* it was stopped: not by running out of budget, but by a
`repetition` condition that noticed the identical call. We build that in
notebook 05.


## HOW — reading the state directly

In [ ]:
# The state object after a successful run. This is what the agent "knows".
print(with_state.state.show())

In [ ]:
# The two views of the same run, side by side.
state = with_state.state

print("MODEL-READABLE (state.messages) — what gets sent to the LLM:")
for m in state.messages:
    body = str(m.get("content"))[:60].replace("\n", " ")
    extra = f"  +{len(m['tool_calls'])} tool_call(s)" if m.get("tool_calls") else ""
    print(f"  {m['role']:<10} {body}{extra}")

print("\nMACHINE-READABLE (state.observations) — what YOUR CODE reasons over:")
for o in state.observations:
    print(f"  step {o.step}  {o.tool:<24} {o.status.value:<9} {o.summary(50)}")

print("\nWhy both? Try answering these from `messages` alone:")
print("  consecutive errors :", state.consecutive_errors())
print("  repeated calls     :", state.repeated_calls() or "none")
print("  context size       :", state.approx_tokens(), "tokens")

## WHAT — the cost curve nobody warns you about

Look at the `context_tokens` figure in each step of a trace. It **rises every
step**, because every step re-sends the entire transcript.

So a 10-step agent run does not cost 10× a 1-step run. It costs roughly the sum
of a growing series — closer to quadratic than linear. This is why step budgets
are a cost control and not merely a safety rail, and it is why "just let it keep
trying" is such an expensive instinct.


In [ ]:
# Watch the context grow, step by step.
outcome = Agent().run(
    "Check the status of order ACME-1043 and tell me whether I can get a refund for changed_mind."
)
print(f"{'step':<6}{'context tokens in':>18}   decision")
running = []
for s in outcome.trace.steps:
    running.append(s.context_tokens)
    print(f"{s.step:<6}{s.context_tokens:>18}   {s.decision[:52]}")
print(f"\nTotal tokens SENT across the run: ~{sum(running)}")
print(f"A single call with the final context would have been: ~{running[-1]}")
print("The gap is what iteration costs you.")

## HOW (parallel mapping) — LangChain's version of state

LangChain calls the transcript the **agent scratchpad**. `AgentExecutor` builds
it for you: it appends `(action, observation)` pairs and formats them into the
prompt each iteration.

```python
# LangChain, roughly
from langchain.agents import AgentExecutor, create_tool_calling_agent
executor = AgentExecutor(agent=agent, tools=tools, max_iterations=5)
executor.invoke({"input": goal})
```

Same loop, same write-back, hidden behind `.invoke()`. The abstraction is
genuinely useful — and it is *exactly* what you just wrote by hand. Knowing that
is the difference between configuring `max_iterations` by guesswork and choosing
it because you know what each iteration costs.


## Recap

- The loop is **THINK → ACT → OBSERVE → UPDATE**, repeated until a stop condition.
- The model is stateless. **The agent's memory is the transcript you build.**
- Remove the write-back and the agent repeats itself forever — rationally.
- Keep the model-readable transcript separate from your machine-readable record.
- Context grows every step, so **cost grows faster than step count**.

**Next → Notebook 03 (Tools & Schemas):** the loop can now think and remember.
Time to let it act on the world — safely.
